In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

rv = pd.read_csv("rv.csv", index_col=0, parse_dates=True)
feats = pd.read_csv("features.csv", index_col=0, parse_dates=True)
load = lambda f: pd.read_csv(f, index_col=0, header=[0, 1], parse_dates=True)
lin, harx, ml, hyb = load("linear_forecasts.csv"), load("harx_forecasts.csv"), load("ml_forecasts.csv"), load("hybrid_forecasts.csv")

test = rv.loc["2021":]
models = {
    "naive": rv.shift(1), "HAR": lin["HAR"], "ARIMA": lin["ARIMA"], "GARCH": lin["GARCH"],
    "HARX": harx["HARX"], "HAR_all": harx["HAR_all"],
    "RF_own": ml["RF_own"], "RF_all": ml["RF_all"], "LGBM_all": ml["LGBM_all"],
    "RF_log": hyb["RF_log"], "Hybrid": hyb["Hybrid"],
}
models = {k: v.reindex(test.index).clip(lower=0.01) for k, v in models.items()}

In [2]:
def sq_loss(f):
    return (test - f)**2

def qlike(f):
    ratio = test**2 / f**2
    return ratio - np.log(ratio) - 1

rmse_table = pd.DataFrame({k: np.sqrt(sq_loss(f).mean()) for k, f in models.items()}).T
qlike_table = pd.DataFrame({k: qlike(f).mean() for k, f in models.items()}).T
rmse_table.round(4), qlike_table.round(3)

(          bitcoin     oil    gold  silver
 naive      0.3591  0.1923  0.1010  0.2607
 HAR        0.2760  0.1634  0.0834  0.2023
 ARIMA      0.2749  0.1681  0.0878  0.2093
 GARCH      0.3206  0.1684  0.0849  0.2165
 HARX       0.2764  0.1532  0.0773  0.2029
 HAR_all    0.2937  0.1987  0.0788  0.2045
 RF_own     0.2831  0.1716  0.0845  0.2102
 RF_all     0.2787  0.1725  0.0797  0.2077
 LGBM_all   0.2993  0.2085  0.0833  0.2184
 RF_log     0.2812  0.1541  0.0830  0.2171
 Hybrid     0.2862  0.1631  0.0764  0.2015,
           bitcoin    oil   gold  silver
 naive       1.645  0.847  1.361   1.276
 HAR         0.565  0.412  0.505   0.519
 ARIMA       0.692  0.481  0.632   0.652
 GARCH       0.585  0.362  0.444   0.469
 HARX        0.568  0.307  0.408   0.523
 HAR_all     0.871  7.801  0.445   0.716
 RF_own      0.601  0.424  0.490   0.559
 RF_all      0.570  0.341  0.419   0.533
 LGBM_all    0.715  0.547  0.476   0.668
 RF_log      0.794  0.362  0.527   0.746
 Hybrid      0.591  0.329  0.394

In [3]:
def dm_test(loss_a, loss_b):
    d = (loss_a - loss_b).dropna()
    fit = sm.OLS(d, np.ones(len(d))).fit(cov_type="HAC", cov_kwds={"maxlags": 4})
    return fit.tvalues.iloc[0], fit.pvalues.iloc[0]

def dm_table(loss, bench="HARX"):
    stats, pvals = {}, {}
    for name, f in models.items():
        if name == bench:
            continue
        res = {c: dm_test(loss(f)[c], loss(models[bench])[c]) for c in test}
        stats[name] = {c: r[0] for c, r in res.items()}
        pvals[name] = {c: r[1] for c, r in res.items()}
    return pd.DataFrame(stats).T, pd.DataFrame(pvals).T

dm_sq, p_sq = dm_table(sq_loss)
dm_q, p_q = dm_table(qlike)
dm_sq.round(2), p_sq.round(3)

(          bitcoin   oil  gold  silver
 naive        4.54  4.63  4.70    2.01
 HAR         -2.20  1.72  2.11   -0.19
 ARIMA       -0.42  1.55  2.13    0.89
 GARCH        6.65  3.07  2.54    1.48
 HAR_all      2.51  5.03  1.07    0.28
 RF_own       1.32  1.91  2.58    2.36
 RF_all       0.43  2.11  1.51    1.43
 LGBM_all     2.29  3.85  2.53    1.77
 RF_log       0.79  0.23  2.30    1.42
 Hybrid       2.18  2.12 -0.45   -0.37,
           bitcoin    oil   gold  silver
 naive       0.000  0.000  0.000   0.044
 HAR         0.028  0.085  0.035   0.847
 ARIMA       0.675  0.121  0.033   0.371
 GARCH       0.000  0.002  0.011   0.138
 HAR_all     0.012  0.000  0.283   0.783
 RF_own      0.187  0.057  0.010   0.018
 RF_all      0.666  0.035  0.132   0.154
 LGBM_all    0.022  0.000  0.011   0.077
 RF_log      0.430  0.815  0.021   0.155
 Hybrid      0.029  0.034  0.655   0.712)

In [4]:
stress = (feats["vix"].shift(1) >= 25).reindex(test.index, fill_value=False)
calm = pd.DataFrame({k: np.sqrt(sq_loss(f)[~stress].mean()) for k, f in models.items()}).T
crisis = pd.DataFrame({k: np.sqrt(sq_loss(f)[stress].mean()) for k, f in models.items()}).T
stress.sum(), calm.round(4), crisis.round(4)

(np.int64(38),
           bitcoin     oil    gold  silver
 naive      0.3243  0.1764  0.1020  0.2673
 HAR        0.2576  0.1515  0.0822  0.2076
 ARIMA      0.2573  0.1498  0.0855  0.2145
 GARCH      0.3013  0.1578  0.0855  0.2198
 HARX       0.2579  0.1391  0.0768  0.2066
 HAR_all    0.2698  0.1760  0.0785  0.2067
 RF_own     0.2636  0.1600  0.0833  0.2143
 RF_all     0.2571  0.1450  0.0798  0.2115
 LGBM_all   0.2729  0.1790  0.0828  0.2220
 RF_log     0.2633  0.1396  0.0822  0.2224
 Hybrid     0.2612  0.1410  0.0766  0.2042,
           bitcoin     oil    gold  silver
 naive      0.5417  0.2784  0.0935  0.2096
 HAR        0.3797  0.2295  0.0914  0.1611
 ARIMA      0.3744  0.2620  0.1024  0.1685
 GARCH      0.4309  0.2283  0.0807  0.1920
 HARX       0.3807  0.2277  0.0812  0.1751
 HAR_all    0.4233  0.3136  0.0809  0.1881
 RF_own     0.3925  0.2366  0.0920  0.1795
 RF_all     0.3972  0.3006  0.0794  0.1791
 LGBM_all   0.4408  0.3500  0.0869  0.1922
 RF_log     0.3829  0.2304  0.0882  0.